# D01 — Model Deployment: From Notebook to Production API

## Why deployment matters for data scientists

A model that lives only in a notebook is not a data product — it's a research artifact. Senior DS roles increasingly require understanding how a model moves from training to production. You don't need to be a DevOps engineer, but you must be able to:
- Serialize a model correctly
- Wrap it in an API
- Validate inputs
- Handle errors gracefully
- Version your artifacts

## Stack

- **joblib**: model serialization (faster than pickle for numpy arrays)
- **FastAPI**: modern, type-safe REST API framework
- **Pydantic**: data validation via Python type hints
- **httpx**: async HTTP client for testing

**Reference:** [FastAPI docs](https://fastapi.tiangolo.com/) | [joblib docs](https://joblib.readthedocs.io/)


In [ ]:
import numpy as np
import pandas as pd
import joblib
import json
import os
import time
import hashlib
from pathlib import Path
from sklearn.datasets import fetch_california_housing, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Model artifacts directory
ARTIFACTS_DIR = Path('/tmp/model_artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Dataset: credit risk (binary classification)
credit_raw = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto').frame
credit = credit_raw.copy()
credit['credit_amount'] = pd.to_numeric(credit['credit_amount'], errors='coerce')
credit['duration'] = pd.to_numeric(credit['duration'], errors='coerce')
credit['age'] = pd.to_numeric(credit['age'], errors='coerce')
credit['target'] = (credit['class'] == 'good').astype(int)

cat_cols = credit.select_dtypes(include='object').columns.drop('class').tolist()
num_cols = credit.select_dtypes(include='number').columns.drop('target').tolist()

X = credit[num_cols + cat_cols]
y = credit['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Credit dataset: {credit.shape}")
print(f"Features: {len(num_cols)} numeric + {len(cat_cols)} categorical")
print(f"Artifacts dir: {ARTIFACTS_DIR}")

---
## Exercise 1 — Model Serialization & Versioning

**Task:** Build a production-grade model serialization system.

1. Train a full sklearn pipeline (preprocessing + LightGBM).
2. Implement `ModelArtifact` dataclass that stores:
   - `model`: the fitted pipeline
   - `feature_names`: list of expected input features
   - `feature_dtypes`: dict of feature → dtype
   - `training_metrics`: dict (AUC, F1, etc.)
   - `model_version`: string `'v{YYYYMMDD}_{hash[:8]}'`
   - `training_data_hash`: MD5 hash of training data (reproducibility check)
   - `sklearn_version`, `python_version`
3. Implement `save_artifact(artifact, path)` and `load_artifact(path)` using joblib.
4. Implement `validate_artifact(artifact)`: verify all fields present, model has `predict_proba`, versions match current environment.

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Optional
import sklearn
import sys
from datetime import datetime

@dataclass
class ModelArtifact:
    model: Any
    feature_names: list
    feature_dtypes: dict
    training_metrics: dict
    model_version: str
    training_data_hash: str
    sklearn_version: str
    python_version: str
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    description: str = ''

def compute_data_hash(df: pd.DataFrame) -> str:
    """MD5 hash of a DataFrame for reproducibility checking."""
    # YOUR CODE HERE
    pass

def make_model_version(metrics: dict) -> str:
    """
    Create version string: 'v{YYYYMMDD}_{auc_4digits}'
    Example: 'v20240115_0742'
    """
    # YOUR CODE HERE
    pass

def save_artifact(artifact: ModelArtifact, path: Path) -> Path:
    """
    Save ModelArtifact to path using joblib.
    Returns the saved path.
    """
    # YOUR CODE HERE
    pass

def load_artifact(path: Path) -> ModelArtifact:
    """
    Load ModelArtifact from path.
    """
    # YOUR CODE HERE
    pass

def validate_artifact(artifact: ModelArtifact) -> dict:
    """
    Validate artifact completeness and environment compatibility.
    Returns dict: {check_name: passed (bool), message: str}
    """
    # YOUR CODE HERE
    # Check: model has predict_proba, feature_names not empty,
    # sklearn version matches, required fields present
    pass

# Build and train pipeline
preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)
])
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('clf', lgb.LGBMClassifier(n_estimators=100, random_state=42, verbosity=-1))
])
pipeline.fit(X_train, y_train)
test_auc = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])

# YOUR CODE HERE: create and save artifact
artifact = None

In [ ]:
# --- ASSERTIONS ---
assert artifact is not None
assert isinstance(artifact, ModelArtifact)
assert artifact.feature_names == list(X_train.columns)
assert 'auc' in artifact.training_metrics or 'test_auc' in artifact.training_metrics
assert artifact.model_version.startswith('v')
assert len(artifact.training_data_hash) == 32  # MD5 hex

# Save and reload
save_path = ARTIFACTS_DIR / f'{artifact.model_version}.joblib'
saved_path = save_artifact(artifact, save_path)
assert saved_path.exists()

loaded = load_artifact(saved_path)
assert loaded.model_version == artifact.model_version
assert loaded.training_data_hash == artifact.training_data_hash

# Validate
validation = validate_artifact(loaded)
assert isinstance(validation, dict)
assert any(v['passed'] if isinstance(v, dict) else v for v in validation.values())

print(f"✓ Exercise 1 passed")
print(f"Model version: {artifact.model_version}")
print(f"Saved to: {saved_path}")
print(f"Test AUC: {test_auc:.4f}")

---
## Exercise 2 — Input Validation with Pydantic

**Concept:** In production, your API receives raw JSON. It could be malformed, have missing fields, have wrong types, or have out-of-range values. Pydantic validates this before it reaches your model.

**Task:** Build Pydantic models for the credit risk prediction API.

1. `CreditApplication`: input schema with all features, proper types, validators.
2. `PredictionResponse`: output schema with prediction, probability, model_version, timestamp.
3. `BatchPredictionRequest`: list of applications with max size 1000.
4. Validators:
   - `credit_amount` must be > 0
   - `duration` must be between 1 and 72 (months)
   - `age` must be between 18 and 100
5. Test with valid and invalid inputs — confirm validation errors are raised correctly.

In [ ]:
from pydantic import BaseModel, Field, validator, ValidationError
from typing import List, Optional
from datetime import datetime
from enum import Enum

# Inspect valid values for categorical features
for col in cat_cols:
    print(f"{col}: {sorted(credit[col].dropna().unique())[:5]}...")

In [ ]:
class CreditApplication(BaseModel):
    """
    Input schema for credit risk prediction.
    All fields match the credit-g dataset features.
    """
    # Numeric features
    credit_amount: float = Field(..., gt=0, description="Credit amount in DM")
    duration: int = Field(..., ge=1, le=72, description="Duration in months")
    age: int = Field(..., ge=18, le=100, description="Applicant age")
    installment_commitment: float = Field(..., ge=1, le=4)
    residence_since: float = Field(..., ge=1, le=4)
    existing_credits: float = Field(..., ge=1, le=4)
    num_dependents: float = Field(..., ge=1, le=2)

    # Categorical features — YOUR CODE HERE
    # Add all categorical columns with appropriate types
    # Use Optional[str] for fields that might be missing
    checking_status: Optional[str] = None
    # ... add remaining categoricals

    class Config:
        extra = 'forbid'  # reject unknown fields

class RiskLevel(str, Enum):
    LOW = 'low'
    MEDIUM = 'medium'
    HIGH = 'high'

class PredictionResponse(BaseModel):
    """
    Output schema for a single prediction.
    """
    # YOUR CODE HERE
    prediction: int           # 0 or 1
    probability_good: float   # P(good credit)
    risk_level: RiskLevel     # low/medium/high
    model_version: str
    prediction_id: str        # UUID or hash
    timestamp: str

class BatchPredictionRequest(BaseModel):
    applications: List[CreditApplication] = Field(..., max_items=1000)
    return_probabilities: bool = True

In [ ]:
# --- ASSERTIONS ---
# Valid input should parse correctly
valid_input = {
    'credit_amount': 5000.0,
    'duration': 24,
    'age': 35,
    'installment_commitment': 2.0,
    'residence_since': 3.0,
    'existing_credits': 1.0,
    'num_dependents': 1.0,
}
app = CreditApplication(**valid_input)
assert app.credit_amount == 5000.0
assert app.age == 35

# Invalid inputs must raise ValidationError
invalid_cases = [
    {'credit_amount': -100},   # negative
    {'duration': 0},            # too short
    {'age': 15},                # underage
    {'age': 150},               # impossible
]
for inv in invalid_cases:
    bad_input = {**valid_input, **inv}
    try:
        CreditApplication(**bad_input)
        print(f"WARNING: Should have rejected {inv}")
    except ValidationError:
        pass  # expected

print("✓ Exercise 2 passed — Pydantic validation working")

---
## Exercise 3 — Prediction Function with Error Handling

**Task:** Build a robust prediction function that wraps your model.

1. Implement `predict_single(application: CreditApplication, artifact: ModelArtifact) -> PredictionResponse`
2. Implement `predict_batch(request: BatchPredictionRequest, artifact: ModelArtifact) -> List[PredictionResponse]`
3. Both functions must:
   - Convert Pydantic input to DataFrame matching training schema exactly
   - Handle missing categorical values (fill with most common training value)
   - Log prediction: timestamp, input hash, output
   - Catch and re-raise model errors with descriptive messages
4. Implement `compute_risk_level(probability: float) -> RiskLevel`:
   - `< 0.3`: HIGH risk (probability of being 'good' is low)
   - `0.3–0.7`: MEDIUM
   - `>= 0.7`: LOW

In [ ]:
import uuid

def compute_risk_level(prob_good: float) -> RiskLevel:
    """
    Map probability of 'good' credit to risk level.
    """
    # YOUR CODE HERE
    pass

def application_to_dataframe(app: CreditApplication,
                               feature_names: list) -> pd.DataFrame:
    """
    Convert Pydantic model to DataFrame matching training schema.
    Fill missing fields with None (model handles imputation).
    """
    # YOUR CODE HERE
    pass

def predict_single(application: CreditApplication,
                    artifact: ModelArtifact) -> PredictionResponse:
    """
    Run single prediction with full error handling and logging.
    """
    # YOUR CODE HERE
    pass

def predict_batch(request: BatchPredictionRequest,
                   artifact: ModelArtifact) -> list:
    """
    Batch prediction — process all at once for efficiency.
    Returns list of PredictionResponse.
    """
    # YOUR CODE HERE
    pass

# Test prediction
sample_app = CreditApplication(**valid_input)
pred = predict_single(sample_app, loaded)

In [ ]:
# --- ASSERTIONS ---
assert pred is not None
assert isinstance(pred, PredictionResponse)
assert pred.prediction in [0, 1]
assert 0 <= pred.probability_good <= 1
assert isinstance(pred.risk_level, RiskLevel)
assert pred.model_version == loaded.model_version
assert len(pred.prediction_id) > 0

# Risk level consistency
assert compute_risk_level(0.1) == RiskLevel.HIGH
assert compute_risk_level(0.5) == RiskLevel.MEDIUM
assert compute_risk_level(0.9) == RiskLevel.LOW

# Batch prediction
batch_req = BatchPredictionRequest(applications=[sample_app] * 5)
batch_preds = predict_batch(batch_req, loaded)
assert batch_preds is not None and len(batch_preds) == 5

print(f"✓ Exercise 3 passed")
print(f"Prediction: {pred.prediction} | P(good)={pred.probability_good:.4f} | Risk: {pred.risk_level}")

---
## Exercise 4 — FastAPI Application

**Task:** Build a production-ready FastAPI application.

1. Create a FastAPI app with these endpoints:
   - `GET /health` → `{status: 'healthy', model_version: str, uptime_seconds: float}`
   - `GET /model/info` → model metadata (version, training metrics, feature count)
   - `POST /predict` → single prediction
   - `POST /predict/batch` → batch prediction (max 1000)
   - `GET /metrics` → request count, mean latency, error count since startup

2. Add middleware:
   - Request timing: add `X-Process-Time` header to every response
   - Request ID: add `X-Request-ID` header

3. The app loads the model artifact at startup (not per request).

4. Write the app to `/tmp/credit_api.py` — it must be importable.

In [ ]:
# Write the FastAPI app to a file
app_code = '''
# credit_api.py — Production Credit Risk API

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from typing import List, Optional
from enum import Enum
import numpy as np
import pandas as pd
import joblib
import uuid
import time
from datetime import datetime
from pathlib import Path

# ── App initialization ────────────────────────────────────────────────────────
app = FastAPI(
    title="Credit Risk API",
    description="Predicts credit default risk using LightGBM",
    version="1.0.0"
)

# YOUR CODE HERE:
# 1. Global state: loaded_artifact, request_count, error_count, latencies, start_time
# 2. @app.on_event('startup'): load artifact from /tmp/model_artifacts/
# 3. Middleware: add X-Process-Time and X-Request-ID headers
# 4. All 5 endpoints
'''

# YOUR CODE HERE: write complete app
full_app_code = """
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from fastapi.middleware.base import BaseHTTPMiddleware
from pydantic import BaseModel, Field, validator
from typing import List, Optional
from enum import Enum
import numpy as np
import pandas as pd
import joblib
import uuid
import time
from datetime import datetime
from pathlib import Path

app = FastAPI(title='Credit Risk API', version='1.0.0')

# Global state
_artifact = None
_request_count = 0
_error_count = 0
_latencies = []
_start_time = time.time()

# YOUR CODE HERE: complete all endpoints and middleware
"""

with open('/tmp/credit_api.py', 'w') as f:
    f.write(full_app_code)
print("App code written to /tmp/credit_api.py")

In [ ]:
# --- ASSERTIONS ---
import importlib.util
spec = importlib.util.spec_from_file_location('credit_api', '/tmp/credit_api.py')
api_module = importlib.util.module_from_spec(spec)
try:
    spec.loader.exec_module(api_module)
    app_obj = api_module.app
    assert app_obj is not None
    # Check routes exist
    routes = {r.path for r in app_obj.routes}
    required_routes = {'/health', '/model/info', '/predict', '/predict/batch', '/metrics'}
    missing = required_routes - routes
    if missing:
        print(f"Missing routes: {missing}")
    else:
        print(f"✓ Exercise 4 passed — All routes present: {sorted(routes)}")
except Exception as e:
    print(f"App import error (expected if code not complete): {e}")

---
## Exercise 5 — API Testing

**Task:** Test the API using FastAPI's built-in test client and httpx.

1. Use `fastapi.testclient.TestClient` to test all endpoints.
2. Write tests for:
   - `GET /health` returns 200 and correct schema
   - `POST /predict` with valid input returns correct schema
   - `POST /predict` with invalid input returns 422 (Unprocessable Entity)
   - `POST /predict/batch` with 100 valid inputs returns 100 predictions
   - `GET /metrics` returns request counts
3. Implement `run_load_test(n_requests=100)`: send 100 sequential requests and measure latency distribution (p50, p95, p99).

In [ ]:
from fastapi.testclient import TestClient

# Try to import the app for testing
try:
    from importlib import reload
    app_test = api_module.app
    client = TestClient(app_test)
except Exception as e:
    print(f"Could not create test client: {e}")
    client = None

def run_api_tests(client) -> dict:
    """
    Run test suite against the API.
    Returns dict of test_name: passed (bool)
    """
    if client is None:
        return {'client_unavailable': False}

    results = {}

    # YOUR CODE HERE: implement each test case
    # Test 1: health check
    # Test 2: valid predict
    # Test 3: invalid predict (422)
    # Test 4: batch predict
    # Test 5: metrics endpoint

    return results

def run_load_test(client, n_requests: int = 50) -> dict:
    """
    Sequential load test.
    Returns: p50, p95, p99 latency in ms, requests_per_second
    """
    # YOUR CODE HERE
    pass

test_results = run_api_tests(client)
load_results = run_load_test(client, n_requests=50) if client else None

In [ ]:
# --- ASSERTIONS ---
assert test_results is not None
print("API Test Results:")
for test, passed in test_results.items():
    status = '✓' if passed else '✗'
    print(f"  {status} {test}")

if load_results is not None:
    print(f"\nLoad Test ({50} requests):")
    for metric, value in load_results.items():
        print(f"  {metric}: {value}")

print("✓ Exercise 5 passed")

---
## Exercise 6 — Model Monitoring: Data Drift Detection

## The Math

In production, the input distribution shifts over time (data drift) and model performance degrades. You need to detect this before users notice.

**Population Stability Index (PSI):**
$$\text{PSI} = \sum_i (\text{actual}_i - \text{expected}_i) \cdot \ln\frac{\text{actual}_i}{\text{expected}_i}$$

- PSI < 0.1: no drift
- 0.1–0.2: moderate drift, investigate
- PSI > 0.2: significant drift, retrain

**Kolmogorov-Smirnov test:** non-parametric test for distribution shift. Significant p-value → distributions differ.

**Task:** Implement a drift monitoring system.

In [ ]:
from scipy import stats

def compute_psi(expected: np.ndarray, actual: np.ndarray,
                 n_bins: int = 10) -> float:
    """
    Population Stability Index.
    expected: reference distribution (training data)
    actual: new data distribution
    Returns PSI value.
    """
    # YOUR CODE HERE
    # 1. Create bins from expected data (percentiles)
    # 2. Compute proportions in each bin for both distributions
    # 3. Clip to avoid log(0): replace 0s with small epsilon
    # 4. Apply PSI formula
    pass

def detect_feature_drift(X_reference: pd.DataFrame,
                          X_new: pd.DataFrame,
                          alpha: float = 0.05) -> pd.DataFrame:
    """
    Detect drift in each feature using KS test + PSI.
    Returns DataFrame: feature, ks_statistic, ks_pvalue, psi, drift_detected
    """
    # YOUR CODE HERE
    # For each numeric feature: KS test + PSI
    # drift_detected = ks_pvalue < alpha OR psi > 0.2
    pass

def monitor_prediction_drift(y_proba_reference: np.ndarray,
                               y_proba_new: np.ndarray) -> dict:
    """
    Monitor drift in model output distribution.
    Returns: psi, ks_stat, ks_pvalue, mean_shift, std_shift
    """
    # YOUR CODE HERE
    pass

# Simulate drift: new data has shifted credit_amount distribution
X_reference = X_train[num_cols].copy()
X_drifted = X_test[num_cols].copy()
X_drifted['credit_amount'] = X_drifted['credit_amount'] * 1.5  # inject drift

drift_report = detect_feature_drift(X_reference, X_drifted)

In [ ]:
# --- ASSERTIONS ---
# PSI test
same_psi = compute_psi(X_reference['credit_amount'].values,
                        X_reference['credit_amount'].values)
assert same_psi is not None
if same_psi is not None:
    assert same_psi < 0.1, "Same distribution should have low PSI"

shifted_psi = compute_psi(X_reference['credit_amount'].values,
                           X_drifted['credit_amount'].values)
if shifted_psi is not None:
    assert shifted_psi > same_psi, "Shifted distribution must have higher PSI"

# Drift report
assert drift_report is not None
assert len(drift_report) == len(num_cols)
assert 'drift_detected' in drift_report.columns
# credit_amount drift should be detected
ca_drift = drift_report[drift_report['feature'] == 'credit_amount']['drift_detected'].values
if len(ca_drift) > 0:
    assert ca_drift[0], "Credit amount drift should be detected"

print(f"✓ Exercise 6 passed")
print(drift_report.to_string(index=False))

---
## Exercise 7 — A/B Model Testing Framework

**Task:** Build an A/B testing framework for comparing two model versions in production.

1. Train two model versions: `model_v1` (LogisticRegression) and `model_v2` (LightGBM).
2. Implement `ABModelRouter` that:
   - Routes `traffic_split`% of traffic to model_v2, rest to model_v1
   - Deterministic assignment: same user_id always gets same model (via hash)
   - Logs: model_version, prediction, probability, timestamp for each request
3. After N predictions, compute `ab_test_summary`: conversion rate per model, significance test, winner.
4. Simulate 500 predictions and report results.

In [ ]:
class ABModelRouter:
    """
    Routes traffic between two model versions for A/B testing.
    Uses consistent hashing for deterministic assignment.
    """
    def __init__(self, model_a: ModelArtifact, model_b: ModelArtifact,
                  traffic_split_b: float = 0.5):
        self.model_a = model_a       # control
        self.model_b = model_b       # treatment
        self.traffic_split_b = traffic_split_b
        self.prediction_log = []     # list of prediction records

    def _assign_model(self, user_id: str) -> str:
        """
        Deterministic assignment: hash user_id to a/b.
        Returns 'a' or 'b'.
        """
        # YOUR CODE HERE
        # Use hashlib.md5(user_id.encode()).hexdigest()
        # Convert first 8 hex chars to int, mod 100, compare to threshold
        pass

    def predict(self, user_id: str,
                 application: CreditApplication) -> dict:
        """
        Route to correct model, make prediction, log result.
        Returns prediction dict with model_version field.
        """
        # YOUR CODE HERE
        pass

    def ab_test_summary(self) -> dict:
        """
        Compute A/B test results from prediction log.
        Returns: n_a, n_b, mean_proba_a, mean_proba_b,
                 ks_stat, ks_pvalue, winner
        """
        # YOUR CODE HERE
        # KS test on prediction probabilities between groups
        pass

# Train two model versions
pipe_v1 = Pipeline([
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
pipe_v1.fit(X_train, y_train)

# YOUR CODE HERE: create both ModelArtifacts, instantiate router, simulate 200 predictions
ab_router = None
ab_summary = None

In [ ]:
# --- ASSERTIONS ---
assert ab_router is not None
assert ab_summary is not None

if isinstance(ab_summary, dict) and 'n_a' in ab_summary:
    # Roughly 50/50 split
    total = ab_summary['n_a'] + ab_summary['n_b']
    split_ratio = ab_summary['n_b'] / total
    assert 0.35 <= split_ratio <= 0.65, "Split should be roughly 50/50"

    # Deterministic: same user_id should always get same model
    user = 'test_user_123'
    assignment1 = ab_router._assign_model(user)
    assignment2 = ab_router._assign_model(user)
    assert assignment1 == assignment2, "Assignment must be deterministic"

print(f"✓ Exercise 7 passed")
if isinstance(ab_summary, dict):
    print(f"A/B test: n_a={ab_summary.get('n_a')}, n_b={ab_summary.get('n_b')}")
    print(f"Winner: {ab_summary.get('winner', 'TBD')}")

---
## Exercise 8 — Model Retraining Pipeline

**Task:** Build an automated retraining pipeline triggered by data drift.

1. Implement `RetrainingPipeline` that:
   - Accepts new data batches via `add_batch(X_new, y_new)`
   - Computes rolling drift score (PSI) on numeric features
   - Triggers retraining when PSI > 0.2 for any feature OR when N new samples accumulate
   - Retrains model on original_data + new_data
   - Validates new model: AUC must improve or stay within 2% of current
   - Saves new artifact with incremented version
2. Simulate 5 data batches with increasing drift.
3. Return `retraining_log`: list of dicts showing when retraining was triggered and outcome.

In [ ]:
class RetrainingPipeline:
    """
    Automated model retraining triggered by data drift or data volume.
    """
    def __init__(self, current_artifact: ModelArtifact,
                  X_train: pd.DataFrame, y_train: pd.Series,
                  X_val: pd.DataFrame, y_val: pd.Series,
                  drift_threshold: float = 0.2,
                  n_samples_threshold: int = 200):
        self.current_artifact = current_artifact
        self.X_train = X_train.copy()
        self.y_train = y_train.copy()
        self.X_val = X_val
        self.y_val = y_val
        self.drift_threshold = drift_threshold
        self.n_samples_threshold = n_samples_threshold
        self.new_X_buffer = []
        self.new_y_buffer = []
        self.retraining_log = []

    def add_batch(self, X_new: pd.DataFrame,
                   y_new: pd.Series) -> dict:
        """
        Add new data batch, check for drift, retrain if needed.
        Returns: {drift_detected: bool, retrained: bool, new_auc: float}
        """
        # YOUR CODE HERE
        pass

    def _retrain(self) -> ModelArtifact:
        """
        Retrain model on original + buffered data.
        Returns new ModelArtifact if AUC doesn't degrade, else None.
        """
        # YOUR CODE HERE
        pass

# Simulate
X_val_rt, X_holdout, y_val_rt, y_holdout = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42
)
retrain_pipeline = RetrainingPipeline(
    loaded, X_train, y_train, X_val_rt, y_val_rt
)
retraining_log = []

In [ ]:
# Simulate 5 batches with increasing drift
np.random.seed(99)
for batch_i in range(5):
    drift_factor = 1 + batch_i * 0.3  # increasing drift
    X_batch = X_holdout.copy()
    X_batch['credit_amount'] = X_batch['credit_amount'] * drift_factor
    result = retrain_pipeline.add_batch(X_batch, y_holdout)
    if result is not None:
        retraining_log.append({'batch': batch_i, **result})
        print(f"Batch {batch_i}: drift={result.get('drift_detected')}, retrained={result.get('retrained')}")

# --- ASSERTIONS ---
assert len(retraining_log) >= 0  # may be 0 if pipeline not fully implemented
if len(retraining_log) > 0:
    assert all('batch' in r for r in retraining_log)
    print(f"✓ Exercise 8 passed — {sum(r.get('retrained', False) for r in retraining_log)} retraining(s) triggered")
else:
    print("Exercise 8: implement add_batch and _retrain to see results")

---
## Exercise 9 — Model Explainability Endpoint

**Task:** Add an explainability endpoint to the API — return SHAP values or feature contributions for each prediction.

1. Implement `explain_prediction(application: CreditApplication, artifact: ModelArtifact) -> dict`:
   - Compute SHAP values using `lgb_model.predict(X, pred_contrib=True)` (last value = bias)
   - Return top 5 features driving the prediction (by |SHAP| value) with their direction and magnitude
   - Format as human-readable: `[{feature, value, impact, direction: 'increases'/'decreases' risk}]`

2. Add `POST /explain` endpoint to the API file.

3. Test: verify explanation features are valid feature names, impacts sum to approximately log-odds.

In [ ]:
def explain_prediction(application: CreditApplication,
                        artifact: ModelArtifact) -> dict:
    """
    Generate feature-level explanation for a single prediction.
    Returns: prediction, probability, explanation (list of top features)
    """
    # YOUR CODE HERE
    # 1. Convert to DataFrame
    # 2. Run through preprocessor only (get encoded features)
    # 3. Use lgb model's pred_contrib=True
    # 4. Map back to original feature names
    # 5. Return top 5 by |SHAP|
    pass

explanation = explain_prediction(sample_app, loaded)

In [ ]:
# --- ASSERTIONS ---
assert explanation is not None
assert 'prediction' in explanation
assert 'probability' in explanation

if 'top_features' in explanation:
    top = explanation['top_features']
    assert len(top) <= 5
    assert all('feature' in f for f in top)
    assert all('impact' in f for f in top)
    assert all(f['direction'] in ('increases', 'decreases') for f in top if 'direction' in f)

print(f"✓ Exercise 9 passed")
print(f"Prediction: {explanation.get('prediction')} | P(good)={explanation.get('probability', 0):.4f}")
if 'top_features' in explanation:
    print("Top influential features:")
    for feat in explanation['top_features']:
        print(f"  {feat}")

---
## Exercise 10 — Capstone: Production-Ready ML System

**Spec:** Assemble everything into a complete, documented, tested ML system.

Build a complete `MLSystem` class that encapsulates:
1. **Training**: full pipeline with feature engineering
2. **Serialization**: versioned artifact with metadata
3. **Serving**: prediction function with input validation
4. **Monitoring**: PSI drift detection on incoming batches
5. **Retraining**: trigger + retrain + promote logic
6. **A/B testing**: route between two model versions
7. **Explainability**: per-prediction SHAP explanations

Then write a complete integration test that:
- Trains the system
- Makes 50 predictions
- Simulates drift
- Triggers retraining
- Compares old vs new model
- Produces a system health report

Return `system_report`: dict with all metrics.

In [ ]:
class MLSystem:
    """
    Complete ML system: train → serve → monitor → retrain.
    """
    def __init__(self, artifacts_dir: Path = ARTIFACTS_DIR):
        self.artifacts_dir = artifacts_dir
        self.current_artifact = None
        self.challenger_artifact = None
        self.prediction_log = []
        self.drift_log = []

    def train(self, X: pd.DataFrame, y: pd.Series,
               X_val: pd.DataFrame, y_val: pd.Series) -> 'MLSystem':
        """Train and register initial model."""
        # YOUR CODE HERE
        pass

    def predict(self, application: CreditApplication,
                 user_id: str = None) -> PredictionResponse:
        """Single prediction with logging."""
        # YOUR CODE HERE
        pass

    def monitor_batch(self, X_batch: pd.DataFrame) -> dict:
        """Check for drift in new data batch."""
        # YOUR CODE HERE
        pass

    def system_health_report(self) -> dict:
        """Complete system health summary."""
        # YOUR CODE HERE
        pass

# YOUR CODE HERE: integration test
system_report = None

In [ ]:
# --- ASSERTIONS ---
assert system_report is not None
required_keys = ['model_version', 'n_predictions', 'drift_detected',
                  'current_auc', 'system_status']
for k in required_keys:
    assert k in system_report, f"Missing: {k}"

assert system_report['system_status'] in ('healthy', 'degraded', 'critical')
assert system_report['current_auc'] > 0.60

print(f"✓ Exercise 10 passed — Full ML system complete")
print(f"\nSystem Health Report:")
for k, v in system_report.items():
    print(f"  {k}: {v}")